### Imports

In [43]:
import os
from dotenv import load_dotenv
from pathlib import Path

# Walk up from notebook dir until we find the project root containing .env.local
_dir = Path.cwd()
while not (_dir / ".env.local").exists() and _dir != _dir.parent:
    _dir = _dir.parent
env_path = _dir / ".env.local"
print("Using:", env_path, "exists:", env_path.exists())

load_dotenv(env_path, override=True)

# Increase DeepEval timeouts for slow cloud models (defaults: 180s per task, 180s per attempt)
os.environ["DEEPEVAL_PER_TASK_TIMEOUT_SECONDS_OVERRIDE"] = "600"
os.environ["DEEPEVAL_PER_ATTEMPT_TIMEOUT_SECONDS_OVERRIDE"] = "300"

from langchain_ollama import ChatOllama
from deepeval.test_case import LLMTestCase
from deepeval.metrics import SummarizationMetric
from deepeval import evaluate, metrics
from deepeval.dataset import EvaluationDataset, Golden
from deepeval.test_case import LLMTestCase
import pandas as pd
from deepeval.models import OllamaModel
from deepeval.evaluate import AsyncConfig

CLOUD_MODEL_BASE_URL = os.getenv("CLOUD_MODEL_BASE_URL")
LOCAL_MODEL_BASE_URL = os.getenv("LOCAL_MODEL_BASE_URL")
OLLAMA_API_KEY = os.getenv("OLLAMA_API_KEY")

Using: /Users/michelecandolfo/Documents/workspaces/DeepEval/ai-engineering-portfolio/.env.local exists: True


### Initialise the Judge

In [44]:
import re
from deepeval.models import OllamaModel

class CleanOllamaModel(OllamaModel):
    """OllamaModel that strips markdown code blocks from LLM responses."""
    
    @staticmethod
    def _strip_markdown_json(text: str) -> str:
        return re.sub(r'^```(?:json)?\s*\n?', '', text.strip()).rstrip('`').strip()

    def generate(self, prompt, schema=None):
        result, cost = super().generate(prompt, schema)
        if isinstance(result, str):
            result = self._strip_markdown_json(result)
            if schema:
                result = schema.model_validate_json(result)
        return result, cost

    async def a_generate(self, prompt, schema=None):
        result, cost = await super().a_generate(prompt, schema)
        if isinstance(result, str):
            result = self._strip_markdown_json(result)
            if schema:
                result = schema.model_validate_json(result)
        return result, cost

In [45]:
judgeModel = CleanOllamaModel(
    model="qwen3-next:80b-cloud",
    base_url=CLOUD_MODEL_BASE_URL,
    temperature=0.0,
    headers={"Authorization": f"Bearer {OLLAMA_API_KEY}"},   
)

### Initialise the Candidate

In [46]:
candidateModel = ChatOllama(
    base_url=CLOUD_MODEL_BASE_URL,
    model="kimi-k2.6:cloud",
    temperature=0.0,
    headers={"Authorization": f"Bearer {OLLAMA_API_KEY}"},  
)

###  Creating Test Data for Test Cases/Goldens

In [47]:
test_data = [
  {
    "input": (
      "Summarize the following text in 2-3 sentences:\n\n"
      "Integration testing verifies that modules and services work correctly when combined. "
      "It focuses on interfaces, data contracts, and error handling across boundaries. "
      "Effective integration tests run in realistic environments (e.g., staging with real databases or well-behaved test doubles) "
      "and help catch issues that unit tests miss, such as serialization errors, mismatched schemas, or race conditions. "
      "They typically come after unit tests and before system tests, providing confidence that the composition of parts behaves as intended."
    )
  },
  {
    "input": (
      "Summarize the following text in 2-3 sentences:\n\n"
      "Regression testing is executed after changes—like bug fixes, refactors, or dependency upgrades—to ensure existing behavior remains intact. "
      "Teams usually automate a critical subset to balance coverage and runtime, prioritizing high-risk flows and past incident areas. "
      "A reliable regression suite reduces release anxiety, shortens feedback cycles, and prevents the reintroduction of previously resolved defects."
    )
  },
  {
    "input": (
      "Summarize the following text in 2-3 sentences:\n\n"
      "Exploratory testing combines learning, test design, and execution in one activity. "
      "Testers use charters and heuristics to investigate software behavior, adapting based on findings rather than following rigid scripts. "
      "This approach often uncovers edge cases, usability problems, and gaps in requirements that scripted testing may overlook."
    )
  }
]

In [48]:
goldens = [Golden(input=d["input"]) for d in test_data]
dataset = EvaluationDataset(goldens=goldens)

### Optional: Push the Goldens Data Set to Confident AI (for reuse with different LLMs)

In [22]:
dataset.push("Summarization Vorlesung")

✅ Dataset successfully pushed to Confident AI! View at 
]8;id=120406;https://app.confident-ai.com/project/cme1cbz3902nx419rccdjpuxv/datasets/cmpcs0ysz0004mz13v8ag868d\https://app.confident-ai.com/project/cme1cbz3902nx419rccdjpuxv/datasets/cmpcs0ysz0004mz13v8ag868d]8;;\

### Optional: Pull the current dataset and convert it into LLMTestCases
##### This is only needed if you already have a dataset in Confident AI and want to evalute different LLMs with it

In [57]:
#dataset.pull(alias="Summarization Dataset", auto_convert_goldens_to_test_cases=True) 

### Add actual output from the candidate LLM to the Goldens Data Set

In [49]:
for g in dataset.goldens:
    response = candidateModel.invoke(g.input)
    g.actual_output = getattr(response, "content", str(response))


#### Convert Goldens to LLMTestCases for Evaluation via DeepEval

In [50]:
test_cases = [
    LLMTestCase(
        input=g.input,
        actual_output=getattr(g, "actual_output", None)
    )
    for g in dataset.goldens
]

### Define metric

##### The summarization metric breaks the score into alignment_score and coverage_score.

<img src="images/Summarization.png" alt="Summarization" width="800">

##### The final score is the minumum of:

- alignment_score which determines whether the summary contains hallucinated or contradictory information to the original text.
- coverage_score which determines whether the summary contains the necessary information from the original text.


#### How to interpret the Summarization Score

| **Score Range** | **Meaning** | **Example Behavior** |
|------------------|-------------|-----------------------|
| **0.8 → 1.0** | 🟢 **Excellent summary** | Faithful to the source, includes all key points, concise and accurate |
| **0.6 → 0.8** | 🟡 **Good but imperfect** | Generally accurate, but missing minor details or slightly redundant |
| **0.3 → 0.6** | 🟠 **Weak summary** | Misses several key points, too short/long, or partially inaccurate |
| **0.0 → 0.3** | 🔴 **Poor summary** | Hallucinated or incorrect info, unrelated to source text |

In [51]:
metric = SummarizationMetric(model=judgeModel)

### Execute evaluation

In [52]:
results = evaluate(test_cases=test_cases, metrics=[metric])

✨ You're running DeepEval's latest Summarization Metric! (using qwen3-next:80b-cloud (Ollama), strict=False, 
async_mode=True)...

Output()



Metrics Summary

  - ✅ Summarization (score: 0.8, threshold: 0.5, strict: False, evaluation model: qwen3-next:80b-cloud (Ollama), reason: The score is 0.80 because the summary accurately reflects the original text without contradictions or extra information but fails to address the question about integration tests running in realistic environments like staging with real databases., error: None)

For test case:

  - input: Summarize the following text in 2-3 sentences:

Integration testing verifies that modules and services work correctly when combined. It focuses on interfaces, data contracts, and error handling across boundaries. Effective integration tests run in realistic environments (e.g., staging with real databases or well-behaved test doubles) and help catch issues that unit tests miss, such as serialization errors, mismatched schemas, or race conditions. They typically come after unit tests and before system tests, providing confidence that the composition of parts behaves a

⚠ WARNING: No hyperparameters logged.
» ]8;id=798102;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Done 🎉! View results on 
]8;id=770038;https://app.confident-ai.com/project/cme1cbz3902nx419rccdjpuxv/test-runs/cmpczns410000o4137nfra9z1/test-cases\https://app.confident-ai.com/project/cme1cbz3902nx419rccdjpuxv/test-runs/cmpczns410000o4137nfra9z1/test-cases]8;;\

#### Display results in a pandas dataframe

In [53]:
rows = []
for tr in results.test_results:
    for m in tr.metrics_data:
        rows.append({
            "test_case": tr.name,
            "input": tr.input,
            "expected_output": tr.expected_output,
            "actual_output": tr.actual_output,
            "metric": m.name,
            "score": m.score,
            "threshold": m.threshold,
            "success": m.success,
            "reason": m.reason,
            "evaluation_model": m.evaluation_model,
            "evaluation_cost": m.evaluation_cost,
        })

results_df = pd.DataFrame(rows)
display(results_df)

,test_case,input,expected_output,actual_output,metric,score,threshold,success,reason,evaluation_model,evaluation_cost
0,test_case_0,Summarize the following text in 2-3 sentences:...,None,Integration testing verifies that modules and ...,Summarization,0.8,0.5,True,The score is 0.80 because the summary accurate...,qwen3-next:80b-cloud (Ollama),0.0
1,test_case_1,Summarize the following text in 2-3 sentences:...,None,Regression testing verifies that existing func...,Summarization,1.0,0.5,True,The score is 1.00 because the summary contains...,qwen3-next:80b-cloud (Ollama),0.0
2,test_case_2,Summarize the following text in 2-3 sentences:...,None,"Exploratory testing merges learning, test desi...",Summarization,1.0,0.5,True,The score is 1.00 because there are no contrad...,qwen3-next:80b-cloud (Ollama),0.0


#### Evaluate the results and add a suggestion for improvements

In [54]:
with pd.option_context("display.max_colwidth", None):
    failing = results_df[results_df["success"].astype(str).str.lower().eq("false")]
    display(failing)

,test_case,input,expected_output,actual_output,metric,score,threshold,success,reason,evaluation_model,evaluation_cost
